In [1]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)   



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="db/chroma_db",  
)

In [3]:
import os
from dotenv import load_dotenv
load_dotenv() 
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [11]:
def ReformulateQuery(user_query,chat_history):
    prompt=f""" Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do NOT answer the question, just reformulate it if needed and otherwise return it as is.
    user query:{user_query}
    chat history:{chat_history}"""
    chat_completion = client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
        )
    return chat_completion.choices[0].message.content

In [ ]:
chat_history={}
def history_aware_chatbot():
    cnt=1
    while True:
        user_query=input("Enter your query")
        if(user_query=='quit'):
            break
        reformulated_query=user_query
        if len(chat_history)!=0:
            reformulated_query=ReformulateQuery(user_query,chat_history)

        results=vector_store.similarity_search(reformulated_query,k=10)
        context_for_llm = "\n\n".join([doc.page_content for doc in results])
        prompt=f"""
        Answer the question based on the given context.
        context: {context_for_llm}
        question: {reformulated_query}
        """
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "user", "content": prompt}
            ],
            model="llama-3.3-70b-versatile",
        )
        chat_history[cnt]={'query':reformulated_query,'response':chat_completion.choices[0].message.content}
        cnt=cnt+1
        print(chat_completion.choices[0].message.content)



In [35]:
history_aware_chatbot()

The second point made about Kishore Kumar is from the perspective of Amitabh Bachchan, who stated: "Hindi film superstar Amitabh Bachchan called Kishore Kumar a multitalented genius who shall remain a phenomenal star." 

This statement highlights Kishore Kumar's exceptional talents and versatility as an artist. Amitabh Bachchan, a renowned actor in the Hindi film industry, acknowledges Kishore Kumar's unique abilities and predicts that he will continue to be remembered as a phenomenal star. This tribute underscores Kishore Kumar's enduring impact on the entertainment industry and his lasting legacy. 

It also implies that Kishore Kumar was not only an incredible singer but also a talented individual with a wide range of skills, which made him a standout figure in the entertainment industry. This statement showcases the respect and admiration that Amitabh Bachchan has for Kishore Kumar, and reinforces the idea that Kishore Kumar was a one-of-a-kind artist.
